In [58]:
from gliner import GLiNER

# Load a GLiNER model
model = GLiNER.from_pretrained("HinoEiji/GLiNER-phobert-large").to("cuda")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

In [65]:
# Sample text for entity prediction
text = "Giá thành thấp chất son lại rất Ok , k có mùi nặng hoá chất"

# Define labels for entity extraction
labels = [
    "thương hiệu",
    "giao hàng",
    "chung",
    "giá cả",
    "sản phẩm",
    "thiết kế sản phẩm",
    "đặc trưng sản phẩm",
    "chất lượng sản phẩm",
    "công dụng sản phẩm",
    "dịch vụ"
]

# Perform entity prediction
entities = model.predict_entities(text, labels, threshold=0.9)

# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])

k có mùi nặng hoá chất => chất lượng sản phẩm


In [60]:
def build_offsets(tokens):
    offsets = []
    current = 0

    for token in tokens:
        start = current
        end = start + len(token)  # ⚠️ end EXCLUSIVE

        offsets.append((start, end))

        current = end + 1  # +1 cho dấu cách

    return offsets

def char_to_bio(text, entities):
    tokens = text.split()
    offsets = build_offsets(tokens)

    tags = ["O"] * len(tokens)

    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]

        for i, (tok_start, tok_end) in enumerate(offsets):

            # không overlap
            if tok_end <= start or tok_start >= end:
                continue

            # token nằm trong span
            if tok_start >= start and tok_end <= end:
                if tok_start == start:
                    tags[i] = f"B-{label}"
                else:
                    tags[i] = f"I-{label}"

    return tokens, tags

In [61]:
from seqeval.metrics.sequence_labeling import get_entities
ent_map = {
    "PRODUCT" : "sản phẩm",
    "PRODUCT FEATURE" : "đặc trưng sản phẩm",
    "PRODUCT USAGE" : "công dụng sản phẩm",
    "PRODUCT QUALITY" : "chất lượng sản phẩm",
    "PRODUCT DESIGN" : "thiết kế sản phẩm",
    "PRICE" : "giá cả",
    "SERVICE" : "dịch vụ",
    "BRANDING" : "thương hiệu",
    "GENERAL" : "chung",
    "DELIVERY" : "giao hàng"
}
def read_conll_file(file_path, ent_path = None):
    data = []
    
    tokens = []
    tags = []

    if ent_path:
        try:
            with open(ent_path, "r", encoding="utf-8") as f:
                list_ent = [line.strip() for line in f if line.strip()]
                list_ent = [ent_map[ent] for ent in list_ent]
        except:
            list_ent = None
    else:
        list_ent = None
    
    if "train" not in file_path:
        list_ent = None
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            # gặp dòng trống => kết thúc 1 sample
            if not line:
                if tokens:
                    data.append(convert_to_span(tokens, tags, list_ent))
                    tokens, tags = [], []
                continue
            
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            
            token, tag = parts
            tokens.append(token)
            tags.append(tag)
    
    # case file không kết thúc bằng dòng trống
    if tokens:
        data.append(convert_to_span(tokens, tags, list_ent))
    
    return data

def convert_to_span(tokens, tags, list_ent = None):
    entities = get_entities(tags)
    new_tags = []
    for tag in tags:
        if tag == "O":
            new_tags.append(tag)
            continue
        splits = tag.split("-")
        new_tags.append(splits[0]+"-"+ent_map[splits[1]])
    
    ner = []
    ner_labels = []

    for ent_type, start, end in entities:
        ent_type = ent_map[ent_type]
        ner.append([start,end, ent_type])
        ner_labels.append(ent_type)
    if list_ent:
        ner_labels = set(ner_labels)
        ner_negatives = set(list_ent) - ner_labels

        return{
            "tokenized_text": tokens,
            "ner": ner,
            "ner_labels": list(ner_labels),
            "ner_negatives": list(ner_negatives),
            "tags" : new_tags
        }
        

    return {
        "tokenized_text": tokens,
        "ner": ner,
        "tags" : new_tags
    }

In [70]:
gt=read_conll_file("/kaggle/working/GLiNER/custom_train_data/v3.3.1/test.txt")

In [ ]:
# y_true = []
# y_pred = []
# for i in range(len(gt)):
#     text = " ".join(gt[i]['tokenized_text'])
#     entities = model.predict_entities(text, labels, threshold=0.9)
#     _, pred = char_to_bio(text, entities)
#     gt[i]['pred'] = pred
#     y_true.append(gt[i]['tags'])
#     y_pred.append(pred)

In [67]:
import json
with open("/kaggle/working/results-GLiNER-phobert-large_threshold_0.9.json", "r") as f:
    prediction = json.load(f)
    preds = prediction['my_data']['predictions']

In [74]:
gt[0]['tokenized_text'][0:6+1]

['đã', 'dùng', 'và', 'thấy', 'rất', 'hài', 'lòng,']

In [77]:
y_true = []
y_pred = []
for i in range(len(gt)):
    tokens = gt[i]["tokenized_text"]
    pred_tag = ["O"]*len(tokens)
    for pred in preds[i]:
        pred_tag[pred["start"]]="B-"+pred['label']
        I = ["I-"+pred['label']]*(pred["end"]-pred["start"])
        pred_tag[pred["start"]+1 : pred["end"]+1] = I
    y_true.append(gt[i]['tags'])
    y_pred.append(pred_tag)
        

In [78]:
from seqeval.metrics import classification_report

In [79]:
print(classification_report(y_true, y_pred, zero_division=0))

                     precision    recall  f1-score   support

              chung       0.48      0.42      0.44       312
chất lượng sản phẩm       0.58      0.26      0.36       320
 công dụng sản phẩm       0.47      0.26      0.33       531
            dịch vụ       0.37      0.24      0.29       180
          giao hàng       0.49      0.50      0.50       450
             giá cả       0.33      0.51      0.40       145
           sản phẩm       0.43      0.24      0.31       313
  thiết kế sản phẩm       0.52      0.47      0.49       332
        thương hiệu       0.04      0.11      0.06        45
 đặc trưng sản phẩm       0.26      0.21      0.23       283

          micro avg       0.42      0.34      0.38      2911
          macro avg       0.40      0.32      0.34      2911
       weighted avg       0.45      0.34      0.38      2911



In [80]:
y_true_flat = [tag for sent in y_true for tag in sent]
y_pred_flat = [tag for sent in y_pred for tag in sent]

In [81]:
from sklearn.metrics import classification_report

print(classification_report(y_true_flat, y_pred_flat))

                       precision    recall  f1-score   support

              B-chung       0.49      0.43      0.46       312
B-chất lượng sản phẩm       0.65      0.29      0.40       320
 B-công dụng sản phẩm       0.55      0.30      0.39       531
            B-dịch vụ       0.38      0.25      0.30       180
          B-giao hàng       0.54      0.55      0.55       450
             B-giá cả       0.38      0.57      0.45       145
           B-sản phẩm       0.51      0.29      0.37       313
  B-thiết kế sản phẩm       0.61      0.55      0.58       332
        B-thương hiệu       0.04      0.11      0.06        45
 B-đặc trưng sản phẩm       0.31      0.25      0.28       283
              I-chung       0.48      0.29      0.36      1390
I-chất lượng sản phẩm       0.65      0.22      0.33      1334
 I-công dụng sản phẩm       0.58      0.25      0.35      2634
            I-dịch vụ       0.38      0.21      0.27      1030
          I-giao hàng       0.50      0.44      0.47  